# QLoRA Support Routing: Controlled Comparison
Runs only in a **hosted Google Colab CUDA GPU runtime**. Do not connect a local runtime. No paid API or automatic paid fallback. Free GPU access is not guaranteed.

Compare TF-IDF, frozen MiniLM, SetFit, untuned Qwen3-0.6B and QLoRA Qwen on the same new 120-case synthetic challenge. It has no independent human review. Training/results are pending until these cells finish.

Select **Runtime > Change runtime type > free GPU (e.g. T4)**, then run cells in order. Upload only `colab_bundle.zip`, not private documents or credentials.


In [ ]:
import sys
from pathlib import Path
try:
    import google.colab
except ImportError:
    raise RuntimeError('Hosted Google Colab only. Do not run on your Mac.')
if not Path('/content').is_dir():
    raise RuntimeError('Use a hosted Colab runtime, not a local runtime.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Stop; choose an available free GPU runtime. Do not purchase compute.')
print('GPU:', torch.cuda.get_device_name(0))
print('Nothing has been installed or downloaded by this notebook yet.')


## Upload the prepared experiment bundle
This contains existing public-model baseline weights and synthetic examples. It contains no live tickets, secrets, or application database.


In [ ]:
from google.colab import files
import io
import stat
import zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only colab_bundle.zip')
name, content = next(iter(uploaded.items()))
if not name.endswith('.zip'):
    raise ValueError('Expected a ZIP bundle')
root = Path('/content/qlora_experiment')
if root.exists():
    raise RuntimeError('Folder already exists. Use a fresh runtime to avoid mixing experiments.')
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    if sum(i.file_size for i in archive.infolist()) > 500 * 1024**2:
        raise ValueError('Unexpectedly large archive')
    for member in archive.infolist():
        target = (root / member.filename).resolve()
        if root.resolve() not in target.parents or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError('Unsafe ZIP entry')
    archive.extractall(root)
sys.path.insert(0, str(root))
from protocol import verify_bundle
manifest = verify_bundle(root)
print(manifest['provenance'])


## Install dependencies in Colab only
Keeps Colab's existing CUDA PyTorch. Downloads are confined to this hosted runtime. There is no Hugging Face login, hub upload or paid inference service.


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root/'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, str(root/'experiment.py'), 'check'], check=True)


## Train and evaluate
Fixed rank-16 NF4 QLoRA, up to 200 optimizer steps. Completion-only loss, validation-loss checkpointing, and validation-only calibration. Results may favor a smaller baseline.

All validation selections are frozen before challenge scoring. Do not tune on challenge errors and call the same challenge independent again. This never changes the application's active router.


In [ ]:
results = Path('/content/qlora_results')
subprocess.run([sys.executable, str(root/'experiment.py'), 'run', '--out', str(results)], check=True)
print((results/'RESULTS.md').read_text())


## Download actual results
Includes the adapter, training evidence, classification metrics, per-domain recall, per-case predictions, memory measurements and latency. Resume claims must come from these completed artifacts. GPU memory and CPU RSS are separate measures; hardware differs between baselines and Qwen.


In [ ]:
archive_path = Path('/content/qlora_results.zip')
subprocess.run([sys.executable, str(root/'export_results.py'), str(results), str(archive_path)], check=True)
files.download(str(archive_path))


References: [PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization), [Qwen model card](https://huggingface.co/Qwen/Qwen3-0.6B), [Colab limits](https://research.google.com/colaboratory/faq.html). No LLM arbiter is included.
